# ClaimsIQ 00 — Snowflake Setup & Seed Data

**Project:** ClaimsIQ — Snowflake-Powered Fraud & Claims Intelligence Agent
**This notebook:** Connects to your real Snowflake account, creates the five-table
schema, and generates realistic synthetic data — including Ananya Rao's
scenario, built explicitly so every downstream notebook can find and reason
about it.

### Before you run this
You need a working Snowflake account with a warehouse you can write to.
Set these as environment variables (never hardcode credentials in a
notebook cell):

```bash
export SNOWFLAKE_ACCOUNT="your_account_identifier"      # e.g. xy12345.ap-south-1
export SNOWFLAKE_USER="your_username"
export SNOWFLAKE_PASSWORD="your_password"
export SNOWFLAKE_WAREHOUSE="COMPUTE_WH"                  # or your warehouse name
export SNOWFLAKE_DATABASE="CLAIMSIQ"
export SNOWFLAKE_SCHEMA="PUBLIC"
export SNOWFLAKE_ROLE="ACCOUNTADMIN"                      # or a role with CREATE privileges
```

If your account uses key-pair auth instead of a password, see the
alternate connection cell in Step 1 — use whichever matches your setup.

## Step 1 — Install & connect

In [1]:
%pip install -q snowflake-connector-python faker pandas "pyarrow>=14,<24" python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import snowflake.connector
from dotenv import load_dotenv

load_dotenv()

required = ["SNOWFLAKE_ACCOUNT", "SNOWFLAKE_USER", "SNOWFLAKE_PASSWORD",
            "SNOWFLAKE_WAREHOUSE", "SNOWFLAKE_DATABASE", "SNOWFLAKE_SCHEMA"]
missing = [v for v in required if not os.environ.get(v)]
assert not missing, f"Missing environment variables: {missing}"

conn = snowflake.connector.connect(
    account=os.environ["SNOWFLAKE_ACCOUNT"],
    user=os.environ["SNOWFLAKE_USER"],
    password=os.environ["SNOWFLAKE_PASSWORD"],
    warehouse=os.environ["SNOWFLAKE_WAREHOUSE"],
    database=os.environ["SNOWFLAKE_DATABASE"],
    schema=os.environ["SNOWFLAKE_SCHEMA"],
    role=os.environ.get("SNOWFLAKE_ROLE"),
    session_parameters={'PYTHON_CONNECTOR_QUERY_RESULT_FORMAT':'JSON'},
)
cs = conn.cursor()
cs.execute("SELECT CURRENT_ROLE()")
print(cs.fetchone())

print("Connected to Snowflake:"+os.environ["SNOWFLAKE_ACCOUNT"], cs.execute("SELECT CURRENT_VERSION()").fetchone()[0])

('ACCOUNTADMIN',)
Connected to Snowflake:DUHJWSA-UA99694 10.30.101


### Alternate: key-pair authentication

If your account requires key-pair auth instead of a password, use this
cell INSTEAD of the one above (set `SNOWFLAKE_PRIVATE_KEY_PATH` and
`SNOWFLAKE_PRIVATE_KEY_PASSPHRASE` as env vars instead of
`SNOWFLAKE_PASSWORD`):

```python
from cryptography.hazmat.primitives import serialization

with open(os.environ["SNOWFLAKE_PRIVATE_KEY_PATH"], "rb") as key_file:
    p_key = serialization.load_pem_private_key(
        key_file.read(),
        password=os.environ.get("SNOWFLAKE_PRIVATE_KEY_PASSPHRASE", "").encode() or None,
    )
pkb = p_key.private_bytes(
    encoding=serialization.Encoding.DER,
    format=serialization.PrivateFormat.PKCS8,
    encryption_algorithm=serialization.NoEncryption(),
)

conn = snowflake.connector.connect(
    account=os.environ["SNOWFLAKE_ACCOUNT"],
    user=os.environ["SNOWFLAKE_USER"],
    private_key=pkb,
    warehouse=os.environ["SNOWFLAKE_WAREHOUSE"],
    database=os.environ["SNOWFLAKE_DATABASE"],
    schema=os.environ["SNOWFLAKE_SCHEMA"],
)
```

## Step 2 — Create the database and schema (if they don't exist)

In [3]:
cs.execute("USE ROLE ACCOUNTADMIN")
cs.execute(f"CREATE DATABASE IF NOT EXISTS {os.environ['SNOWFLAKE_DATABASE']}")
cs.execute(f"CREATE SCHEMA IF NOT EXISTS {os.environ['SNOWFLAKE_DATABASE']}.{os.environ['SNOWFLAKE_SCHEMA']}")
cs.execute(f"USE DATABASE {os.environ['SNOWFLAKE_DATABASE']}")
cs.execute(f"USE SCHEMA {os.environ['SNOWFLAKE_SCHEMA']}")
print("Database and schema ready.")

Database and schema ready.


## Step 3 — Create the five tables

In [4]:
import os
print(repr(os.environ.get("SNOWFLAKE_ROLE")))

'ACCOUNTADMIN'


In [5]:
DDL = """


CREATE OR REPLACE TABLE CUSTOMERS (
    customer_id STRING PRIMARY KEY,
    name STRING,
    email STRING,
    signup_date DATE,
    account_risk_tier STRING,        -- 'low', 'medium', 'high'
    verified_devices STRING          -- comma-separated device IDs
);

CREATE OR REPLACE TABLE ORDERS (
    order_id STRING PRIMARY KEY,
    customer_id STRING,
    product_id STRING,
    product_name STRING,
    order_value NUMBER(10,2),
    order_ts TIMESTAMP_NTZ,
    device_id STRING,
    shipping_country STRING,
    is_festive_sale BOOLEAN
);

CREATE OR REPLACE TABLE CLAIMS (
    claim_id STRING PRIMARY KEY,
    order_id STRING,
    claim_reason STRING,
    claim_amount NUMBER(10,2),
    filed_ts TIMESTAMP_NTZ,
    status STRING                    -- 'pending', 'approved', 'denied', 'escalated'
);

CREATE OR REPLACE TABLE TRANSACTIONS (
    txn_id STRING PRIMARY KEY,
    customer_id STRING,
    amount NUMBER(10,2),
    txn_ts TIMESTAMP_NTZ,
    device_id STRING,
    payment_method STRING,
    txn_status STRING                -- 'approved', 'declined', 'flagged'
);

CREATE OR REPLACE TABLE FRAUD_SIGNALS (
    signal_id STRING PRIMARY KEY,
    customer_id STRING,
    signal_type STRING,              -- 'new_device', 'velocity_spike', 'chargeback_history'
    severity STRING,                 -- 'low', 'medium', 'high'
    detected_ts TIMESTAMP_NTZ
);
"""

for statement in DDL.strip().split(";"):
    if statement.strip():
        cs.execute(statement)

print("All five tables created.")

All five tables created.


## Step 4 — Generate synthetic data with Faker

300 customers, ~1,500 orders, ~400 claims, ~3,000 transactions, ~150 fraud
signals — enough volume that queries return realistic, non-trivial
results, not just the one hand-crafted scenario.

In [6]:
cs.execute("SELECT CURRENT_ROLE()")
print(cs.fetchone())

('ACCOUNTADMIN',)


In [7]:
import uuid
import random
from datetime import datetime, timedelta
from faker import Faker

fake = Faker("en_IN")
random.seed(42)

N_CUSTOMERS = 300
customers = []
for i in range(N_CUSTOMERS):
    signup = fake.date_between(start_date="-5y", end_date="-30d")
    customers.append({
        "customer_id": f"CUST{i:05d}",
        "name": fake.name(),
        "email": fake.email(),
        "signup_date": signup,
        "account_risk_tier": random.choices(["low", "medium", "high"], weights=[80, 15, 5])[0],
        "verified_devices": ",".join([f"DEV{random.randint(1000,9999)}" for _ in range(random.randint(1, 3))]),
    })

PRODUCTS = [
    ("P100", "HomeChef Blender X200", 2499),
    ("P101", "SonicBuds Pro Earbuds", 3999),
    ("P102", "HomeChef 4-Slice Toaster", 899),
    ("P103", "UltraView 55-inch TV", 38000),
    ("P104", "BassMax Home Theater System", 45000),
    ("P105", "AeroFit Smartwatch", 6500),
]

orders = []
for i in range(1500):
    cust = random.choice(customers)
    product = random.choice(PRODUCTS)
    order_ts = datetime.now() - timedelta(days=random.randint(0, 365), hours=random.randint(0, 23))
    orders.append({
        "order_id": f"ORD{i:06d}",
        "customer_id": cust["customer_id"],
        "product_id": product[0],
        "product_name": product[1],
        "order_value": product[2],
        "order_ts": order_ts,
        "device_id": random.choice(cust["verified_devices"].split(",")),
        "shipping_country": "India",
        "is_festive_sale": random.random() < 0.2,
    })

print(f"Generated {len(customers)} customers, {len(orders)} orders.")

Generated 300 customers, 1500 orders.


In [8]:
CLAIM_REASONS = [
    "Motor stopped working", "Speaker not producing sound", "Screen flickering",
    "Battery draining fast", "Item arrived damaged", "Blade assembly loose",
    "Heating element not working", "Left earbud not charging",
]

claims = []
claimable_orders = random.sample(orders, 400)
for i, order in enumerate(claimable_orders):
    filed_ts = order["order_ts"] + timedelta(days=random.randint(1, 20))
    claims.append({
        "claim_id": f"CLM{i:05d}",
        "order_id": order["order_id"],
        "claim_reason": random.choice(CLAIM_REASONS),
        "claim_amount": order["order_value"],
        "filed_ts": filed_ts,
        "status": random.choices(["pending", "approved", "denied"], weights=[60, 30, 10])[0],
    })

transactions = []
for i in range(3000):
    cust = random.choice(customers)
    txn_ts = datetime.now() - timedelta(days=random.randint(0, 90), hours=random.randint(0, 23))
    transactions.append({
        "txn_id": f"TXN{i:06d}",
        "customer_id": cust["customer_id"],
        "amount": round(random.uniform(500, 5000), 2),
        "txn_ts": txn_ts,
        "device_id": random.choice(cust["verified_devices"].split(",")),
        "payment_method": random.choice(["card", "upi", "netbanking"]),
        "txn_status": random.choices(["approved", "declined", "flagged"], weights=[92, 5, 3])[0],
    })

fraud_signals = []
flagged_customers = random.sample(customers, 40)
for i, cust in enumerate(flagged_customers):
    fraud_signals.append({
        "signal_id": f"SIG{i:05d}",
        "customer_id": cust["customer_id"],
        "signal_type": random.choice(["new_device", "velocity_spike", "chargeback_history"]),
        "severity": random.choice(["low", "medium", "high"]),
        "detected_ts": datetime.now() - timedelta(days=random.randint(0, 30)),
    })

print(f"Generated {len(claims)} claims, {len(transactions)} transactions, {len(fraud_signals)} fraud signals.")

Generated 400 claims, 3000 transactions, 40 fraud signals.


## Step 5 — Inject Ananya Rao's scenario explicitly

This is the case every downstream notebook will query. Built by hand so
the signals are exact and reproducible — not left to random chance.

In [9]:
ANANYA_ID = "CUST99001"

# 3-year loyal customer, low risk tier
customers.append({
    "customer_id": ANANYA_ID,
    "name": "Ananya Rao",
    "email": "ananya.rao@example.com",
    "signup_date": (datetime.now() - timedelta(days=3*365)).date(),
    "account_risk_tier": "low",
    "verified_devices": "DEV1111,DEV2222",
})

# The order: a ₹45,000 home theater system, during the festive sale
ananya_order_ts = datetime.now() - timedelta(hours=6)
orders.append({
    "order_id": "ORD99001",
    "customer_id": ANANYA_ID,
    "product_id": "P104",
    "product_name": "BassMax Home Theater System",
    "order_value": 45000,
    "order_ts": ananya_order_ts,
    "device_id": "DEV1111",
    "shipping_country": "India",
    "is_festive_sale": True,
})

# The claim: plausible, filed within the window
claims.append({
    "claim_id": "CLM99001",
    "order_id": "ORD99001",
    "claim_reason": "Speaker not producing sound",
    "claim_amount": 45000,
    "filed_ts": ananya_order_ts + timedelta(hours=2),
    "status": "pending",
})

# 3 flagged transactions in the last 48 hours, across 2 UNRECOGNIZED devices
for i in range(3):
    transactions.append({
        "txn_id": f"TXN9900{i}",
        "customer_id": ANANYA_ID,
        "amount": round(random.uniform(8000, 15000), 2),
        "txn_ts": datetime.now() - timedelta(hours=random.randint(1, 48)),
        "device_id": random.choice(["DEV9998", "DEV9999"]),   # NOT in her verified_devices
        "payment_method": "card",
        "txn_status": "flagged",
    })

# Also add several NORMAL transactions this week (consistent with festive-sale enthusiasm)
for i in range(3, 9):
    transactions.append({
        "txn_id": f"TXN9900{i}",
        "customer_id": ANANYA_ID,
        "amount": round(random.uniform(2000, 6000), 2),
        "txn_ts": datetime.now() - timedelta(hours=random.randint(1, 72)),
        "device_id": "DEV1111",
        "payment_method": "upi",
        "txn_status": "approved",
    })

# Fraud signals: velocity_spike + new_device, both HIGH severity
fraud_signals.append({
    "signal_id": "SIG99001", "customer_id": ANANYA_ID, "signal_type": "velocity_spike",
    "severity": "high", "detected_ts": datetime.now() - timedelta(hours=20),
})
fraud_signals.append({
    "signal_id": "SIG99002", "customer_id": ANANYA_ID, "signal_type": "new_device",
    "severity": "high", "detected_ts": datetime.now() - timedelta(hours=18),
})

print("Ananya Rao's scenario (customer_id = CUST99001) added:")
print(f"  1 order (₹45,000, festive sale), 1 claim (pending),")
print(f"  9 transactions (3 flagged on new devices, 6 normal), 2 HIGH severity fraud signals.")

Ananya Rao's scenario (customer_id = CUST99001) added:
  1 order (₹45,000, festive sale), 1 claim (pending),
  9 transactions (3 flagged on new devices, 6 normal), 2 HIGH severity fraud signals.


## Step 6 — Load everything into Snowflake

In [10]:
import pandas as pd
from snowflake.connector.pandas_tools import write_pandas

def load_table(table_name, records):
    df = pd.DataFrame(records)
    df.columns = [c.upper() for c in df.columns]
    success, nchunks, nrows, _ = write_pandas(conn, df, table_name.upper())
    print(f"{table_name}: loaded {nrows} rows — {'OK' if success else 'FAILED'}")

load_table("CUSTOMERS", customers)
load_table("ORDERS", orders)
load_table("CLAIMS", claims)
load_table("TRANSACTIONS", transactions)
load_table("FRAUD_SIGNALS", fraud_signals)

CUSTOMERS: loaded 301 rows — OK
ORDERS: loaded 1501 rows — OK
CLAIMS: loaded 401 rows — OK
TRANSACTIONS: loaded 3009 rows — OK
FRAUD_SIGNALS: loaded 42 rows — OK


In [11]:
def load_table(table_name, records):
    df = pd.DataFrame(records)
    df.columns = [c.upper() for c in df.columns]
    success, nchunks, nrows, _ = write_pandas(conn, df, table_name.upper())
    print(f"{table_name}: loaded {nrows} rows — {'OK' if success else 'FAILED'}")


load_table("CLAIMS", claims)

CLAIMS: loaded 401 rows — OK


## Step 7 — Verify

In [12]:
for table in ["CUSTOMERS", "ORDERS", "CLAIMS", "TRANSACTIONS", "FRAUD_SIGNALS"]:
    count = cs.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"{table}: {count} rows")

print("\nAnanya Rao's data:")
cs.execute("SELECT * FROM CUSTOMERS WHERE customer_id = 'CUST99001'")
print(cs.fetchone())

cs.execute("""
    SELECT signal_type, severity, TO_VARCHAR(detected_ts) AS detected_ts
    FROM FRAUD_SIGNALS WHERE customer_id = 'CUST99001'
""")
for row in cs.fetchall():
    print(row)

CUSTOMERS: 301 rows
ORDERS: 1501 rows
CLAIMS: 802 rows
TRANSACTIONS: 3009 rows
FRAUD_SIGNALS: 42 rows

Ananya Rao's data:
('CUST99001', 'Ananya Rao', 'ananya.rao@example.com', datetime.date(2023, 8, 26), 'low', 'DEV1111,DEV2222')
('velocity_spike', 'high', '56648823-05-12 19:57:43.000')
('new_device', 'high', '56649051-07-10 03:59:07.000')


## Done — what's next

Schema and data are live in Snowflake. Move to
`01_mcp_server_snowflake.ipynb`, which builds the MCP server that wraps
these tables as tools — the notebook every agent (Day 11 LangGraph,
Lab 19 CrewAI, Lab 20 SWARM) will connect to next.

In [13]:
cs.close()
conn.close()
print("Connection closed. Data persists in Snowflake for the next notebook.")

Connection closed. Data persists in Snowflake for the next notebook.
